# Cricket Range-Map Processing

Turns the SINA range-map GIFs from `Webscraping.ipynb` into two numbers per
species: how much of the map its occurrence dots cover, and how much its
range overlaps with every other species'. Every SINA map shares the same
canvas and legend, so one crop and one colored-vs-grayscale test works for
all of them.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

CRICKETS_DIR = Path.home() / 'Discrete_Signals' / 'Crickets'

# All ~100 maps are the same 960x726 canvas: title text on top, the US
# county-outline map in the middle, then the legend and "Map prepared..."
# caption at the bottom. Checked row by row across every file--map content only
# ever appears in rows 63-636--so this slice keeps the map body and drops the
# title and legend everywhere.
MAP_BODY_ROWS = slice(63, 637)

# A pixel counts as a colored occurrence dot if its RGB channels span more than
# this. The background, the gray county outlines, and the black text are all
# neutral (R == G == B); only the legend colors and their anti-aliased edges
# have real saturation, which also catches dot edges an exact-color match misses.
SATURATION_THRESHOLD = 30

In [ ]:
range_map_paths = sorted(
    path for path in CRICKETS_DIR.rglob('*map*')
    if 'checkpoint' not in str(path)
    and path.suffix.lower() in {'.gif', '.jpg', '.jpeg', '.png'}
)
print(f'{len(range_map_paths)} range maps')

In [ ]:
def occurrence_mask(range_map_path):
    """Boolean array over the cropped map body: True where a pixel is a colored
    occurrence dot rather than background, county outline, or text."""
    with Image.open(range_map_path) as image:
        image.seek(0)                       # first frame, in case it's an animated GIF
        rgb = np.array(image.convert('RGB'))[MAP_BODY_ROWS]
    channel_spread = rgb.max(axis=-1).astype(int) - rgb.min(axis=-1).astype(int)
    return channel_spread > SATURATION_THRESHOLD

In [ ]:
occurrence_by_species = {
    path.parent.name: occurrence_mask(path) for path in range_map_paths
}
# Every mask has the same shape--MAP_BODY_ROWS crops every map identically.
map_height, map_width = next(iter(occurrence_by_species.values())).shape
species_names = list(occurrence_by_species)

## Range size per species

In [ ]:
# Colored pixels as a fraction of the whole cropped map body. That body includes
# some ocean/margin (land can't be told from white background by color alone),
# but the margin is identical for every species, so this is still a fair basis
# for comparing species even though it isn't a true land-area percentage.
range_proportion = pd.Series(
    {name: mask.sum() / (map_height * map_width) for name, mask in occurrence_by_species.items()},
    name='proportion_of_map_occupied',
).sort_values(ascending=False)

range_proportion_pct = (range_proportion * 100).round(2).rename('percent_of_map_occupied')
range_proportion_pct.head(10)

## Pairwise range overlap

In [ ]:
# overlap[A, B] = |A ∩ B| / |A|: the fraction of A's range that lies within B's.
# Asymmetric--if A sits entirely inside a larger B, overlap[A, B] is 1.0 but
# overlap[B, A] is smaller. The diagonal is 1.0 by construction.
#
# Each species' mask is flattened to one row of a (species, pixels) matrix;
# multiplying that matrix by its own transpose gives every pairwise
# intersection count at once (True*True = 1), far faster than looping over pairs.
flat_masks = np.stack([occurrence_by_species[name].ravel() for name in species_names]).astype(np.uint32)
range_sizes = flat_masks.sum(axis=1)
intersection_counts = flat_masks @ flat_masks.T

with np.errstate(invalid='ignore'):        # a species with an empty mask gives 0/0 = nan
    overlap_fraction = intersection_counts / range_sizes[:, None]

overlap_matrix = pd.DataFrame(overlap_fraction, index=species_names, columns=species_names)
overlap_matrix_pct = (overlap_matrix * 100).round(2)

## Save

In [ ]:
overlap_matrix_pct.to_csv(CRICKETS_DIR / 'species_overlap_matrix_pct.csv')
range_proportion_pct.to_csv(CRICKETS_DIR / 'species_range_proportions_pct.csv')
print('wrote species_overlap_matrix_pct.csv and species_range_proportions_pct.csv')

In [ ]:
overlap_matrix_pct